#Repository Clone

In [1]:
import os

print("[INFO] Cloning the official FastHMR repository...")
if not os.path.exists('FastHMR'):
    !git clone https://github.com/TaatiTeam/FastHMR.git

# Enter the project directory
%cd /content/FastHMR
print(f"[INFO] Working directory set to: {os.getcwd()}")

[INFO] Cloning the official FastHMR repository...
Cloning into 'FastHMR'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 185 (delta 25), reused 180 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (185/185), 23.88 MiB | 22.96 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/FastHMR
[INFO] Working directory set to: /content/FastHMR


#Install Dependencies

In [2]:
print("[INFO] Installing base requirements...")
!pip install -q -r requirements.txt

# Install lapx (Modern fork of lap) required for YOLO tracking
print("[INFO] Installing tracking dependencies (lapx)...")
!pip install -q lapx

# Compile PyTorch3D
print("[INFO] Compiling PyTorch3D... Please be patient.")
!pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"

print("[SUCCESS] Dependencies installed successfully.")

[INFO] Installing base requirements...
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 63.4 MB/s eta 0:00:00
[INFO] Installing tracking dependencies (lapx)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.7 MB/s eta 0:00:00
[INFO] Compiling PyTorch3D... Please be patient.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Weights & Body Models Setup

In [3]:
import os
import urllib.parse
import shutil
import getpass
from huggingface_hub import snapshot_download

print("[INFO] 1/4 Downloading FastHMR weights from Hugging Face...")
hf_dir = 'data/weights'
os.makedirs(hf_dir, exist_ok=True)
snapshot_download(repo_id="SoroushMehraban/FastHMR", local_dir=hf_dir)

print("[INFO] 2/4 Restructuring FastHMR folders...")
os.makedirs('checkpoints/feature_extractors', exist_ok=True)
os.makedirs('checkpoints/fasthmr_ckpts', exist_ok=True)
shutil.copy('data/weights/feature_extractors/camerahmr.pth.tr', 'checkpoints/feature_extractors/camerahmr.pth.tr')
for f in os.listdir('data/weights/fasthmr_ckpts'):
    if f.endswith('.tr'):
        shutil.copy(os.path.join('data/weights/fasthmr_ckpts', f), os.path.join('checkpoints/fasthmr_ckpts', f))

print("[INFO] 3/4 Downloading base SMPL 3D body models (Bypassing broken fetch_data.sh)...")
# Download the fallback body_models.tar.gz directly from the authors' public Drive
!wget -q --show-progress "https://drive.usercontent.google.com/download?id=1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ&export=download" -O body_models.tar.gz
!tar -xf body_models.tar.gz

# Download HMR2.0 checkpoint fallback
os.makedirs('checkpoints', exist_ok=True)
!wget -q --show-progress "https://drive.usercontent.google.com/download?id=1J6l8teyZrL0zFzHhzkC7efRhU0ZJ5G9Y&export=download" -O checkpoints/hmr2a.ckpt

print("\n[INFO] 4/4 Downloading Max Planck weights (CameraHMR Dependencies)...")
print("=====================================================================")
print("⚠️ REQUIRED: You need a free account at https://camerahmr.is.tue.mpg.de/")
print("This is a strict license requirement from the Max Planck Institute.")
print("=====================================================================")

# Securely prompt user for credentials
user_email = input("Please enter your CameraHMR registered email: ")
user_pass = getpass.getpass("Please enter your CameraHMR password: ")

username = urllib.parse.quote(user_email)
password = urllib.parse.quote(user_pass)

os.makedirs('checkpoints/body_models', exist_ok=True)

# Download required CameraHMR weights and body models
!wget -q --show-progress --post-data "username={username}&password={password}" 'https://download.is.tue.mpg.de/download.php?domain=camerahmr&sfile=cam_model_cleaned.ckpt' -O 'checkpoints/feature_extractors/cam_model_cleaned.ckpt' --no-check-certificate
!wget -q --show-progress --post-data "username={username}&password={password}" 'https://download.is.tue.mpg.de/download.php?domain=camerahmr&sfile=smpl_mean_params.npz' -O 'checkpoints/body_models/smpl_mean_params.npz' --no-check-certificate
!wget -q --show-progress --post-data "username={username}&password={password}" 'https://download.is.tue.mpg.de/download.php?domain=camerahmr&sfile=SMPL_NEUTRAL.pkl' -O 'checkpoints/body_models/SMPL_NEUTRAL.pkl' --no-check-certificate

# Download OpenMMLab regressor
!wget -q --show-progress "https://openmmlab-share.oss-cn-hangzhou.aliyuncs.com/mmhuman3d/models/J_regressor_h36m.npy?versionId=CAEQHhiBgIDE6c3V6xciIDdjYzE3MzQ4MmU4MzQyNmRiZDA5YTg2YTI5YWFkNjRi" -O 'checkpoints/body_models/J_regressor_h36m.npy'

print("\n[SUCCESS] All weights and 3D body models are fully configured and ready.")

[INFO] 1/4 Downloading FastHMR weights from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[INFO] 2/4 Restructuring FastHMR folders...
[INFO] 3/4 Downloading base SMPL 3D body models (Bypassing broken fetch_data.sh)...
body_models.tar.gz  100%[===================>] 940.94K  --.-KB/s    in 0.1s    
checkpoints/hmr2a.c 100%[===================>]   2.36K  --.-KB/s    in 0s      

[INFO] 4/4 Downloading Max Planck weights (CameraHMR Dependencies)...
⚠️ REQUIRED: You need a free account at https://camerahmr.is.tue.mpg.de/
This is a strict license requirement from the Max Planck Institute.
Please enter your CameraHMR registered email: antonioperearocamora@gmail.com
Please enter your CameraHMR password: ··········
checkpoints/feature 100%[===================>] 767.99M  9.79MB/s    in 74s     
checkpoints/body_mo 100%[===================>]   1.28K  --.-KB/s    in 0s      
checkpoints/body_mo 100%[===================>] 235.73M  8.35MB/s    in 20s     
checkpoints/body_mo 100%[===================>] 915.20K   759KB/s    in 1.2s    

[SUCCESS] All weights and 3D body models are fully co

#Download Real Test Video & Inference

In [4]:
import os

print("[INFO] Downloading a sample video containing people...")
os.makedirs('examples', exist_ok=True)
video_path = 'examples/test_video.mp4'

# Using a reliable public sample video of people walking (commonly used in object detection)
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4"
!wget -q --show-progress {video_url} -O {video_path}

print("[INFO] Launching inference on the test video...")
os.makedirs('visualization', exist_ok=True)
!python demo.py --video {video_path} --output_pth visualization/ --visualize
print("[SUCCESS] Inference finished. Check the 'visualization/' folder for the reconstructed 3D meshes.")

[INFO] Downloading a sample video containing people...
examples/test_video 100%[===================>]   5.75M  --.-KB/s    in 0.05s   
[INFO] Launching inference on the test video...
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Preprocess: Per-frame feature extraction |######                          | 127/647/content/FastHMR/data/preprocess/camerahmr/utils/geometry.py:274: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Pleas

#Advanced Hardware Profiling (VRAM & FPS)

In [10]:
import time
import subprocess
import threading
import cv2

print("--- FastHMR: Advanced Performance & VRAM Benchmark ---")

peak_vram = 0
keep_polling = True

def poll_vram():
    global peak_vram
    while keep_polling:
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
                capture_output=True, text=True
            )
            current_vram = int(result.stdout.strip())
            if current_vram > peak_vram:
                peak_vram = current_vram
        except Exception:
            pass
        time.sleep(0.2)

video_path = 'examples/test_video.mp4'

print("[INFO] Background VRAM monitor ACTIVATED.")
monitor_thread = threading.Thread(target=poll_vram)
monitor_thread.start()

start_time = time.time()
cmd = f"python demo.py --video {video_path} --output_pth benchmark_out/"

print("[INFO] Running benchmark... This will take a moment.")
try:
    subprocess.run(cmd.split(), capture_output=True, text=True)
    end_time = time.time()

    keep_polling = False
    monitor_thread.join()

    total_time = end_time - start_time

    cap = cv2.VideoCapture(video_path)
    num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    fps = num_frames / total_time

    print("\n" + "="*45)
    print("🏆 FINAL PERFORMANCE RESULTS 🏆")
    print("="*45)
    print(f"Total Frames:       {num_frames}")
    print(f"Total Time:         {total_time:.2f} seconds")
    print(f"Throughput:         {fps:.2f} FPS")
    print(f"Peak VRAM Usage:    {peak_vram} MB")
    print("="*45)

except Exception as e:
    keep_polling = False
    print(f"\n[CRITICAL ERROR] Benchmark failed: {e}")

--- FastHMR: Advanced Performance & VRAM Benchmark ---
[INFO] Background VRAM monitor ACTIVATED.
[INFO] Running benchmark... This will take a moment.

🏆 FINAL PERFORMANCE RESULTS 🏆
Total Frames:       647
Total Time:         45.91 seconds
Throughput:         14.09 FPS
Peak VRAM Usage:    5823 MB
